# Machine Learning Analysis

Boosted Decision Tree training, feature importance analysis, and supervised learning evaluation.

In [1]:
# Required imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load pre-computed results
RESULTS_DIR = Path('../docs/output')
df_shape = pd.read_csv(RESULTS_DIR / 'ch2_waveform_shape_features.csv')
print(f'Loaded {len(df_shape)} events with shape features')
print(f'Columns: {list(df_shape.columns)}')

FileNotFoundError: [Errno 2] No such file or directory: '../docs/output/ch2_waveform_shape_features.csv'

# Boosted Decision Tree (BDT) Analysis

Using time-of-flight as ground truth (Δt > 20 ns = neutron), train a BDT to see if combining all pulse shape features provides better discrimination than individual parameters.

In [ ]:
# BDT Training and Evaluation
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

print("="*80)
print("BOOSTED DECISION TREE (BDT) ANALYSIS")
print("="*80)

# Define features and labels
feature_cols = ['rise_time_ns', 'fall_time_ns', 'fwhm_ns', 'peak_amplitude_v', 
                'charge_asymmetry', 'tail_to_peak_ratio', 'baseline_std_v', 'total_charge_v_s']

# Prepare data - drop rows with NaN in any feature
df_bdt = df_shape[feature_cols + ['delta_t_ns']].dropna()

X = df_bdt[feature_cols].values
y = (df_bdt['delta_t_ns'] > 20.0).astype(int).values  # Truth: delta_t > 20 ns = neutron

print(f"\nDataset:")
print(f"  Total events: {len(y)}")
print(f"  Neutrons (Δt > 20 ns): {y.sum()} ({100*y.mean():.1f}%)")
print(f"  Background: {len(y) - y.sum()} ({100*(1-y.mean()):.1f}%)")
print(f"  Features: {feature_cols}")

# Train/test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nTrain set: {len(y_train)} events ({y_train.sum()} neutrons)")
print(f"Test set: {len(y_test)} events ({y_test.sum()} neutrons)")

# Scale features (helps with convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train BDT
print("\nTraining BDT...")
bdt = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    verbose=0
)

bdt.fit(X_train_scaled, y_train)

# Cross-validation score
cv_scores = cross_val_score(bdt, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"Cross-validation AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Predict on test set
y_pred_proba = bdt.predict_proba(X_test_scaled)[:, 1]
y_pred = bdt.predict(X_test_scaled)

# Calculate metrics
test_auc = roc_auc_score(y_test, y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

print(f"\n" + "="*80)
print("BDT PERFORMANCE")
print("="*80)
print(f"\nTest Set AUC: {test_auc:.3f}")

# Compare with individual features
print("\n" + "-"*60)
print("Comparison: BDT vs Individual Features")
print("-"*60)
print(f"{'Method':<25} {'AUC':>8}")
print("-"*35)
print(f"{'BDT (all features)':<25} {test_auc:>8.3f}")

for i, feat in enumerate(feature_cols):
    feat_auc = max(
        roc_auc_score(y_test, X_test[:, i]),
        roc_auc_score(y_test, -X_test[:, i])
    )
    print(f"{feat:<25} {feat_auc:>8.3f}")

improvement = test_auc - 0.648  # Best single feature was rise_time with ~0.648
print("-"*35)
print(f"Improvement over best single: {improvement:+.3f}")

# Classification report
print("\n" + "-"*60)
print("Classification Report (threshold=0.5)")
print("-"*60)
print(classification_report(y_test, y_pred, target_names=['Background', 'Neutron']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}")
print(f"  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}")

In [ ]:
# Plot ROC curves: BDT vs individual features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC curves comparison
ax1 = axes[0]
ax1.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'BDT (AUC={test_auc:.3f})')

colors = plt.get_cmap('tab10')
for i, feat in enumerate(feature_cols[:4]):  # Top 4 features only for clarity
    feat_auc_pos = roc_auc_score(y_test, X_test[:, i])
    feat_auc_neg = roc_auc_score(y_test, -X_test[:, i])
    if feat_auc_neg > feat_auc_pos:
        fpr_f, tpr_f, _ = roc_curve(y_test, -X_test[:, i])
        feat_auc = feat_auc_neg
    else:
        fpr_f, tpr_f, _ = roc_curve(y_test, X_test[:, i])
        feat_auc = feat_auc_pos
    ax1.plot(fpr_f, tpr_f, '--', color=colors(i), alpha=0.7, 
             label=f'{feat} (AUC={feat_auc:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves: BDT vs Individual Features', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: Feature importance
ax2 = axes[1]
importances = bdt.feature_importances_
sorted_idx = np.argsort(importances)
ax2.barh(range(len(feature_cols)), importances[sorted_idx], color='steelblue')
ax2.set_yticks(range(len(feature_cols)))
ax2.set_yticklabels([feature_cols[i] for i in sorted_idx])
ax2.set_xlabel('Feature Importance', fontsize=12)
ax2.set_title('BDT Feature Importance', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
roc_path = RESULTS_DIR / 'bdt_roc_and_importance.png'
plt.savefig(roc_path, dpi=150, bbox_inches='tight')
print(f"\n✓ ROC and importance plot saved to: {roc_path}")
plt.show()

# Print feature importance ranking
print("\n" + "-"*60)
print("Feature Importance Ranking")
print("-"*60)
for idx in sorted_idx[::-1]:
    print(f"  {feature_cols[idx]:<22} {importances[idx]:.3f}")

In [ ]:
# BDT Score Distribution and Efficiency Analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Left: BDT score distribution
ax1 = axes[0]
bins = np.linspace(0, 1, 41)
ax1.hist(y_pred_proba[y_test == 0], bins=bins, alpha=0.6, label='Background', 
         color='blue', density=True)
ax1.hist(y_pred_proba[y_test == 1], bins=bins, alpha=0.6, label='Neutron', 
         color='red', density=True)
# ax1.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Threshold=0.5')
ax1.set_xlabel('BDT Score', fontsize=12)
ax1.set_ylabel('Normalized Counts', fontsize=12)
ax1.set_title('BDT Score Distribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Middle: Efficiency vs threshold
ax2 = axes[1]
thresholds_plot = np.linspace(0, 1, 101)
signal_eff = []
bkg_rej = []
purity = []

for thresh in thresholds_plot:
    pred = (y_pred_proba >= thresh)
    # Signal efficiency (TPR)
    sig_eff = pred[y_test == 1].sum() / (y_test == 1).sum() if (y_test == 1).sum() > 0 else 0
    # Background rejection (1 - FPR)
    bkg_r = 1 - pred[y_test == 0].sum() / (y_test == 0).sum() if (y_test == 0).sum() > 0 else 0
    # Purity
    pur = pred[y_test == 1].sum() / pred.sum() if pred.sum() > 0 else 0
    
    signal_eff.append(sig_eff)
    bkg_rej.append(bkg_r)
    purity.append(pur)

ax2.plot(thresholds_plot, signal_eff, 'b-', linewidth=2, label='Signal Efficiency')
ax2.plot(thresholds_plot, bkg_rej, 'r-', linewidth=2, label='Background Rejection')
ax2.plot(thresholds_plot, purity, 'g--', linewidth=2, label='Purity')
ax2.axvline(0.5, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('BDT Threshold', fontsize=12)
ax2.set_ylabel('Efficiency / Rejection', fontsize=12)
ax2.set_title('Efficiency vs Threshold', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

# Right: Signal efficiency vs purity
ax3 = axes[2]
ax3.plot(signal_eff, purity, 'b-', linewidth=2)
ax3.scatter([signal_eff[50]], [purity[50]], color='red', s=100, zorder=5, 
            label=f'Threshold=0.5\nEff={signal_eff[50]:.2f}, Pur={purity[50]:.2f}')
ax3.set_xlabel('Signal Efficiency', fontsize=12)
ax3.set_ylabel('Purity', fontsize=12)
ax3.set_title('Efficiency vs Purity Trade-off', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 1.05)
ax3.set_ylim(0, 1.05)

plt.tight_layout()
eff_path = RESULTS_DIR / 'bdt_score_and_efficiency.png'
plt.savefig(eff_path, dpi=150, bbox_inches='tight')
print(f"✓ BDT score and efficiency plot saved to: {eff_path}")
plt.show()

# Print working points
print("\n" + "-"*60)
print("BDT Working Points")
print("-"*60)
print(f"{'Threshold':<12} {'Sig. Eff.':<12} {'Bkg. Rej.':<12} {'Purity':<12}")
print("-"*50)
for thresh_val in [0.3, 0.4, 0.5, 0.6, 0.7]:
    idx = int(thresh_val * 100)
    print(f"{thresh_val:<12.1f} {signal_eff[idx]:<12.2f} {bkg_rej[idx]:<12.2f} {purity[idx]:<12.2f}")

In [ ]:
# BDT Summary and Interpretation
print("="*80)
print("BDT ANALYSIS SUMMARY")
print("="*80)

print(f"""
RESULTS:
--------
• BDT AUC: {test_auc:.3f}
• Best single feature (rise_time): ~0.65
• Improvement from combining features: {test_auc - 0.648:+.3f}

INTERPRETATION:
---------------
""")

if test_auc < 0.65:
    print("""• The BDT shows NO significant improvement over individual features.
• Combining pulse shape features does not reveal hidden discrimination power.
• The features are likely correlated and redundant for n/γ separation.
""")
elif test_auc < 0.70:
    print("""• The BDT shows MARGINAL improvement over individual features.
• Combining features extracts slightly more information.
• However, discrimination remains WEAK (AUC < 0.7).
""")
elif test_auc < 0.80:
    print("""• The BDT shows FAIR discrimination (0.70 < AUC < 0.80).
• Combining features provides meaningful improvement.
• Could be useful as a secondary selection cut.
""")
else:
    print("""• The BDT shows GOOD discrimination (AUC > 0.80).
• Combining features significantly improves separation.
• Can be used for effective n/γ discrimination.
""")

print("""CONCLUSION:
-----------
The pulse shape features, even when combined with machine learning,
provide limited discrimination between neutrons and gammas in this
borated scintillator system. The time-of-flight (Δt > 20 ns) remains
the primary and most reliable neutron identification method.

POSSIBLE IMPROVEMENTS:
----------------------
1. Add more features (pulse integral ratios, multiple time windows)
2. Use raw waveform samples as input to neural network
3. Higher sampling rate to resolve fine pulse structure
4. Different scintillator with better intrinsic n/γ PSD
""")

# Save BDT model info
bdt_summary = {
    'test_auc': test_auc,
    'cv_auc_mean': cv_scores.mean(),
    'cv_auc_std': cv_scores.std(),
    'n_train': len(y_train),
    'n_test': len(y_test),
    'features': feature_cols,
    'feature_importances': dict(zip(feature_cols, importances.tolist()))
}

import json
bdt_path = RESULTS_DIR / 'bdt_summary.json'
with open(bdt_path, 'w') as f:
    json.dump(bdt_summary, f, indent=2)
print(f"\n✓ BDT summary saved to: {bdt_path}")

In [ ]:
# Feature Importance Analysis - Multiple Methods
from sklearn.inspection import permutation_importance

print("="*80)
print("DETAILED FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# 1. Built-in feature importance (from BDT)
builtin_importance = bdt.feature_importances_

# 2. Permutation importance (more robust)
print("\nCalculating permutation importance (this may take a moment)...")
perm_importance = permutation_importance(bdt, X_test_scaled, y_test, 
                                          n_repeats=30, random_state=42, n_jobs=-1)

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Top-left: Built-in importance (bar chart)
ax1 = axes[0, 0]
sorted_idx = np.argsort(builtin_importance)
colors = plt.get_cmap('RdYlGn')(np.linspace(0.2, 0.8, len(feature_cols)))
ax1.barh(range(len(feature_cols)), builtin_importance[sorted_idx], 
         color=[colors[i] for i in range(len(feature_cols))])
ax1.set_yticks(range(len(feature_cols)))
ax1.set_yticklabels([feature_cols[i] for i in sorted_idx])
ax1.set_xlabel('Importance Score', fontsize=11)
ax1.set_title('BDT Built-in Feature Importance\n(Gini/Split-based)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Top-right: Permutation importance with error bars
ax2 = axes[0, 1]
perm_sorted_idx = np.argsort(perm_importance.importances_mean)
ax2.barh(range(len(feature_cols)), perm_importance.importances_mean[perm_sorted_idx],
         xerr=perm_importance.importances_std[perm_sorted_idx], 
         color='steelblue', capsize=3)
ax2.set_yticks(range(len(feature_cols)))
ax2.set_yticklabels([feature_cols[i] for i in perm_sorted_idx])
ax2.set_xlabel('Mean Accuracy Decrease', fontsize=11)
ax2.set_title('Permutation Importance\n(with std error bars)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')
ax2.axvline(0, color='black', linestyle='-', linewidth=0.5)

# Bottom-left: Comparison of both methods
ax3 = axes[1, 0]
x_pos = np.arange(len(feature_cols))
width = 0.35

# Normalize both to [0,1] for comparison
builtin_norm = builtin_importance / builtin_importance.max()
perm_norm = perm_importance.importances_mean / perm_importance.importances_mean.max()

# Sort by average of both
avg_imp = (builtin_norm + perm_norm) / 2
final_sort_idx = np.argsort(avg_imp)

ax3.barh(x_pos - width/2, builtin_norm[final_sort_idx], width, 
         label='Built-in (normalized)', color='coral', alpha=0.8)
ax3.barh(x_pos + width/2, perm_norm[final_sort_idx], width,
         label='Permutation (normalized)', color='steelblue', alpha=0.8)
ax3.set_yticks(x_pos)
ax3.set_yticklabels([feature_cols[i] for i in final_sort_idx])
ax3.set_xlabel('Normalized Importance', fontsize=11)
ax3.set_title('Comparison: Built-in vs Permutation', fontsize=12, fontweight='bold')
ax3.legend(loc='lower right')
ax3.grid(True, alpha=0.3, axis='x')

# Bottom-right: Feature importance heatmap/summary
ax4 = axes[1, 1]
# Create importance matrix
imp_matrix = np.column_stack([builtin_norm, perm_norm])
im = ax4.imshow(imp_matrix[final_sort_idx], cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax4.set_yticks(range(len(feature_cols)))
ax4.set_yticklabels([feature_cols[i] for i in final_sort_idx])
ax4.set_xticks([0, 1])
ax4.set_xticklabels(['Built-in', 'Permutation'])
ax4.set_title('Importance Heatmap', fontsize=12, fontweight='bold')

# Add text annotations
for i in range(len(feature_cols)):
    for j in range(2):
        val = imp_matrix[final_sort_idx][i, j]
        color = 'white' if val > 0.5 else 'black'
        ax4.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im, ax=ax4, label='Normalized Importance')

plt.tight_layout()
imp_path = RESULTS_DIR / 'bdt_feature_importance_detailed.png'
plt.savefig(imp_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Detailed importance plot saved to: {imp_path}")
plt.show()

# Print importance ranking table
print("\n" + "="*80)
print("FEATURE IMPORTANCE RANKING")
print("="*80)
print(f"\n{'Rank':<6} {'Feature':<22} {'Built-in':>10} {'Permutation':>12} {'Avg':>8}")
print("-"*62)

for rank, idx in enumerate(final_sort_idx[::-1], 1):
    print(f"{rank:<6} {feature_cols[idx]:<22} {builtin_importance[idx]:>10.4f} "
          f"{perm_importance.importances_mean[idx]:>12.4f} {avg_imp[idx]:>8.3f}")

# Identify most important features
top_features = [feature_cols[i] for i in final_sort_idx[::-1][:3]]
print(f"\n→ Top 3 most important features: {', '.join(top_features)}")